# Projeto LusoEnergia — Parte II: Visualização, Diagnóstico e Redução de Dimensão

Este notebook implementa integralmente os requisitos da **Parte II**, incorporando:
- O uso da **mediana** como medida central robusta (justificada pela assimetria positiva identificada na Parte I) sem remoção artificial dos picos físicos reais.
- A referência explícita da **carga basal constante** (frigorífico, arcas e *standby*) para evidenciar o consumo variável real e os períodos de ausência da habitação.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams['figure.dpi'] = 110

In [ ]:
# 1. Definir janela do grupo
SOMA_DOS_NUMEROS_DE_ALUNO = 101833
rng = np.random.default_rng(SOMA_DOS_NUMEROS_DE_ALUNO)
inicio = pd.Timestamp(rng.choice(pd.date_range('2007-01-01', '2009-05-01', freq='MS')))
fim = inicio + pd.DateOffset(months=18)

# 2. Carregar dados
possiveis_caminhos = [
    '../data/raw/individual+household+electric+power+consumption.zip',
    'data/raw/individual+household+electric+power+consumption.zip',
    '../data/raw/household_power_consumption.zip',
    'data/raw/household_power_consumption.zip'
]
caminho_zip = next((p for p in possiveis_caminhos if os.path.exists(p)), possiveis_caminhos[0])

dtypes = {
    'Date': 'str', 'Time': 'str',
    'Global_active_power': 'float32', 'Global_reactive_power': 'float32',
    'Voltage': 'float32', 'Global_intensity': 'float32',
    'Sub_metering_1': 'float32', 'Sub_metering_2': 'float32', 'Sub_metering_3': 'float32'
}

df = pd.read_csv(caminho_zip, sep=';', na_values=['?'], dtype=dtypes, low_memory=False, compression='zip')
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df.drop(columns=['Date', 'Time'], inplace=True)
df.set_index('datetime', inplace=True)
df = df.loc[inicio:fim].copy()

# 3. Interpolar apenas falhas curtas (<= 60 min), mantendo dias completos em falta como NaN e preservando picos reais
df_clean = df.interpolate(method='time', limit=60)

# 4. Carga Basal Constante (P10) e Decomposição Energética
CARGA_BASAL_KW = float(df_clean['Global_active_power'].quantile(0.10))
ENERGIA_BASAL_KWH_DIA = CARGA_BASAL_KW * 24.0

df_clean['Energia_Total_Wh'] = (df_clean['Global_active_power'] * 1000.0) / 60.0
df_clean['Sub_metering_outros'] = (
    df_clean['Energia_Total_Wh'] - (df_clean['Sub_metering_1'] + df_clean['Sub_metering_2'] + df_clean['Sub_metering_3'])
).clip(lower=0)

def obter_estacao(mes):
    if mes in [12, 1, 2]: return 'Inverno'
    elif mes in [3, 4, 5]: return 'Primavera'
    elif mes in [6, 7, 8]: return 'Verão'
    return 'Outono'

df_clean['Hora'] = df_clean.index.hour
df_clean['Dia_Semana_Num'] = df_clean.index.dayofweek
df_clean['Tipo_Dia'] = np.where(df_clean['Dia_Semana_Num'] >= 5, 'Fim de Semana', 'Dia Útil')
df_clean['Estacao'] = df_clean.index.month.map(obter_estacao)

print(f"Janela: {inicio.strftime('%Y-%m-%d')} a {fim.strftime('%Y-%m-%d')} | Carga Basal Constante: {CARGA_BASAL_KW:.3f} kW ({ENERGIA_BASAL_KWH_DIA:.2f} kWh/dia)")

## 2.1 Análise Descritiva e Diagnóstica: Sazonalidade (com Mediana e Carga Basal) e *Drill-Down*

In [ ]:
# FIGURA 1: Sazonalidade Diária (Mediana, IQR e Linha Constante de Carga Basal)
perfil_horario = df_clean.groupby('Hora')['Global_active_power'].agg(
    mediana='median',
    q25=lambda x: x.quantile(0.25),
    q75=lambda x: x.quantile(0.75)
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(perfil_horario.index, perfil_horario['q25'], perfil_horario['q75'],
                color='#3498db', alpha=0.22, label='Intervalo Interquartil (P25–P75)')
ax.plot(perfil_horario.index, perfil_horario['mediana'], color='#1f618d', linewidth=2.5, marker='o', label='Mediana Horária (kW)')
ax.axhline(CARGA_BASAL_KW, color='#c0392b', linestyle=':', linewidth=1.8,
           label=f'Carga Basal Constante ({CARGA_BASAL_KW:.2f} kW — frigorífico/standby)')

pico_noite_h, pico_noite_v = perfil_horario['mediana'].idxmax(), perfil_horario['mediana'].max()
ax.annotate(f'Pico Noturno ({pico_noite_h}h: {pico_noite_v:.2f} kW)',
            xy=(pico_noite_h, pico_noite_v), xytext=(pico_noite_h - 5.5, pico_noite_v + 0.42),
            arrowprops=dict(arrowstyle='->', color='#1f618d', lw=1.5),
            fontsize=10, fontweight='bold', color='#1f618d')

ax.set_title('A mediana horária revela dois picos ocupacionais (manhã e noite) acima da carga basal fixa',
             fontsize=12.5, fontweight='bold', loc='left', pad=12)
ax.set_xlabel('Hora do Dia (0h – 23h)', fontsize=11)
ax.set_ylabel('Potência Ativa Mediana (kW)', fontsize=11)
ax.set_xticks(range(0, 24))
ax.set_ylim(bottom=0)
ax.legend(frameon=True, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# FIGURA 2: Sazonalidade Semanal (Mediana Diária em kWh/dia + Linha Constante de Carga Basal)
energia_diaria = df_clean['Energia_Total_Wh'].resample('D').sum(min_count=1200) / 1000.0
df_dias = pd.DataFrame({'Energia_kWh': energia_diaria, 'Dia_Num': energia_diaria.index.dayofweek})

nomes_dias = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo']
resumo_semanal = df_dias.groupby('Dia_Num')['Energia_kWh'].median()
cores_barras = ['#5d6d7e']*5 + ['#d35400']*2

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(nomes_dias, resumo_semanal.values, color=cores_barras, width=0.62, edgecolor='black', linewidth=0.6)
ax.set_ylim(0, resumo_semanal.max() * 1.20)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.4, f'{h:.1f} kWh',
            ha='center', va='bottom', fontsize=9.5, fontweight='bold')

ax.axhline(ENERGIA_BASAL_KWH_DIA, color='#c0392b', linestyle=':', linewidth=1.8,
           label=f'Carga Basal Constante ({ENERGIA_BASAL_KWH_DIA:.1f} kWh/dia)')
med_uteis = resumo_semanal.iloc[:5].median()
med_fds = resumo_semanal.iloc[5:].median()
aumento_pct = ((med_fds - med_uteis) / med_uteis) * 100

ax.axhline(med_uteis, color='#2c3e50', linestyle='--', linewidth=1.3, label=f'Mediana Dias Úteis ({med_uteis:.1f} kWh/dia)')
ax.annotate(f'Fim de semana: +{aumento_pct:.1f}% na mediana',
            xy=(5.5, med_fds), xytext=(3.1, resumo_semanal.max() * 1.09),
            arrowprops=dict(arrowstyle='->', color='#d35400', lw=1.5),
            fontsize=10, fontweight='bold', color='#d35400')

ax.set_title(f'A mediana diária sobe {aumento_pct:.0f}% ao fim de semana, assente numa carga basal fixa de {ENERGIA_BASAL_KWH_DIA:.1f} kWh/dia',
             fontsize=12.5, fontweight='bold', loc='left', pad=12)
ax.set_xlabel('Dia da Semana', fontsize=11)
ax.set_ylabel('Energia Mediana Diária (kWh/dia)', fontsize=11)
ax.legend(loc='upper left', fontsize=9.5)
plt.tight_layout()
plt.show()

In [ ]:
# FIGURA 3: Sazonalidade Anual (Mediana Mensal do Consumo Diário por Circuito + Carga Basal)
cols_energia = ['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'Sub_metering_outros']
diario_subs = df_clean[cols_energia].resample('D').sum(min_count=1200) / 1000.0
mensal_subs = diario_subs.resample('MS').median()

labels_meses = [d.strftime('%Y-%m') for d in mensal_subs.index]
nomes_legenda = [
    'Sub 1: Cozinha (forno, placa, m. loiça)',
    'Sub 2: Lavandaria (m. roupa, secar, frigorífico)',
    'Sub 3: Climatização e Águas Quentes (termoacumulador/AC)',
    'Outros: Iluminação, tomadas e equipamentos gerais'
]
cores_subs = ['#e74c3c', '#f39c12', '#2980b9', '#95a5a6']

fig, ax = plt.subplots(figsize=(13, 5.5))
bottom = np.zeros(len(mensal_subs))

for col, nome, cor in zip(cols_energia, nomes_legenda, cores_subs):
    vals = mensal_subs[col].fillna(0).values
    ax.bar(labels_meses, vals, bottom=bottom, label=nome, color=cor, width=0.68, edgecolor='white', linewidth=0.5)
    bottom += vals

ax.axhline(ENERGIA_BASAL_KWH_DIA, color='#c0392b', linestyle='--', linewidth=1.7,
           label=f'Carga Basal Constante ({ENERGIA_BASAL_KWH_DIA:.1f} kWh/dia)')
ax.set_ylim(0, bottom.max() * 1.24)
plt.xticks(rotation=45, ha='right')

idx_max, idx_min = int(np.argmax(bottom)), int(np.argmin(bottom))
ax.annotate(f'Pico Inverno ({labels_meses[idx_max]}: {bottom[idx_max]:.1f} kWh/dia)',
            xy=(idx_max, bottom[idx_max]), xytext=(max(0, idx_max - 2), bottom[idx_max] + 4.0),
            arrowprops=dict(arrowstyle='->', color='#1a5276', lw=1.5),
            fontsize=10, fontweight='bold', color='#1a5276')

ax.annotate(f'Vale Verão ({labels_meses[idx_min]}: {bottom[idx_min]:.1f} kWh/dia)',
            xy=(idx_min, bottom[idx_min]), xytext=(max(0, idx_min - 2), bottom[idx_min] + 6.0),
            arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.5),
            fontsize=10, fontweight='bold', color='#c0392b')

ax.set_title('A mediana diária duplica entre o verão e o inverno devido à climatização e águas quentes',
             fontsize=12.5, fontweight='bold', loc='left', pad=12)
ax.set_xlabel('Mês de Referência', fontsize=11)
ax.set_ylabel('Energia Mediana Diária (kWh/dia)', fontsize=11)
ax.legend(loc='upper right', fontsize=9, frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# FIGURA 4 (SMALL MULTIPLES): Perfil Horário Mediano por Estação e Tipo de Dia + Carga Basal Constante
estacoes_ordem = ['Inverno', 'Primavera', 'Verão', 'Outono']
perfil_estacao = df_clean.groupby(['Estacao', 'Tipo_Dia', 'Hora'])['Global_active_power'].median().reset_index()

fig, axes = plt.subplots(1, 4, figsize=(15, 4.5), sharey=True, sharex=True)
cores_tipo = {'Dia Útil': '#2c3e50', 'Fim de Semana': '#e67e22'}

for ax, est in zip(axes, estacoes_ordem):
    sub = perfil_estacao[perfil_estacao['Estacao'] == est]
    for tipo in ['Dia Útil', 'Fim de Semana']:
        dados_t = sub[sub['Tipo_Dia'] == tipo]
        ax.plot(dados_t['Hora'], dados_t['Global_active_power'], label=tipo, color=cores_tipo[tipo], linewidth=2.2)
    ax.axhline(CARGA_BASAL_KW, color='#c0392b', linestyle=':', linewidth=1.5, label='Carga Basal Constante')
    ax.set_title(est, fontsize=11.5, fontweight='bold')
    ax.set_xlabel('Hora do Dia (0–23h)', fontsize=10)
    ax.set_xticks([0, 6, 12, 18, 23])
    ax.set_ylim(0, perfil_estacao['Global_active_power'].max() * 1.18)

axes[0].set_ylabel('Potência Ativa Mediana (kW)', fontsize=11)
axes[0].annotate('Pico noturno\nmais intenso', xy=(20, 1.8), xytext=(6, 2.15),
                 arrowprops=dict(arrowstyle='->', color='#e67e22', lw=1.3),
                 fontsize=9, fontweight='bold', color='#d35400')
axes[2].annotate('Consumo próximo\nda carga basal', xy=(14, 0.45), xytext=(5, 1.35),
                 arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.3),
                 fontsize=9, fontweight='bold', color='#2c3e50')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', bbox_to_anchor=(0.99, 0.96), frameon=True)
fig.suptitle('Perfis medianos por estação: o consumo variável assenta sobre uma carga basal constante',
             fontsize=12.5, fontweight='bold', x=0.01, ha='left', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# FIGURA 5 (DRILL-DOWN): Agosto de 2009 vs. Carga Basal Constante
agosto_2009 = diario_subs.loc['2009-08-01':'2009-08-31']
fig, ax = plt.subplots(figsize=(13, 4.8))
bottom_ag = np.zeros(len(agosto_2009))
dias_str = [d.strftime('%d/%m') for d in agosto_2009.index]

for col, nome, cor in zip(cols_energia, nomes_legenda, cores_subs):
    vals = agosto_2009[col].fillna(0).values
    ax.bar(dias_str, vals, bottom=bottom_ag, label=nome, color=cor, width=0.72)
    bottom_ag += vals

ax.axhline(ENERGIA_BASAL_KWH_DIA, color='#c0392b', linestyle='--', linewidth=1.8,
           label=f'Carga Basal Constante ({ENERGIA_BASAL_KWH_DIA:.1f} kWh/dia)')
ax.axvspan(7.5, 21.5, color='#f1c40f', alpha=0.16, label='Ausência / Férias (Consumo variável ≈ 0 kWh)')
ax.annotate('Consumo encosta na linha constante\n(apenas frigorífico e standby ativos)',
            xy=(14, ENERGIA_BASAL_KWH_DIA), xytext=(8.5, 18),
            arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.5),
            fontsize=10, fontweight='bold', color='#922b21')

ax.set_ylim(0, max(bottom_ag.max() * 1.25, 25))
plt.xticks(rotation=45, ha='right')
ax.set_title('Drill-down Agosto/2009: a linha basal constante evidencia duas semanas de ausência total',
             fontsize=12.5, fontweight='bold', loc='left', pad=12)
ax.set_xlabel('Dia do Mês (Agosto de 2009)', fontsize=11)
ax.set_ylabel('Energia Diária (kWh/dia)', fontsize=11)
ax.legend(loc='upper right', fontsize=8.8, frameon=True)
plt.tight_layout()
plt.show()

#### Hipóteses Diagnósticas Ordenadas por Plausibilidade
1. **Ausência prolongada por férias de verão (*Plausibilidade Muito Alta*):** Durante ~2 semanas em agosto, o consumo diário encosta exatamente na **linha constante da carga basal** (~5,8 kWh/dia). O consumo variável cai a zero, permanecendo apenas o frigorífico, arca congeladora e *standby* em funcionamento.
2. **Menor necessidade térmica estival (*Plausibilidade Moderada*):** Nos restantes dias ocupados de agosto, temperaturas ambiente elevadas reduzem o acionamento do termoacumulador (`Sub_metering_3`) e eliminam o aquecimento ambiente.
3. **Avaria nos sub-contadores (*Plausibilidade Muito Baixa / Refutada*):** Se os sub-contadores falhassem mas houvesse consumo real, a energia surgiria em `Sub_metering_outros`, o que não acontece.

## 2.2 Perfis Diários com Redução de Dimensão (PCA vs. t-SNE)

In [ ]:
# 1. Construir matriz N_dias x 24 horas usando a MEDIANA horária (robusta a picos de 1 min)
df_hourly = df_clean[['Global_active_power']].copy()
df_hourly['Data'] = df_hourly.index.date
df_hourly['Hora'] = df_hourly.index.hour

matriz_perfis = df_hourly.pivot_table(
    index='Data', columns='Hora', values='Global_active_power', aggfunc='median'
)
dias_antes = len(matriz_perfis)
matriz_perfis.dropna(inplace=True)
print(f"Matriz de perfis diários: {len(matriz_perfis)} dias completos x 24 horas (excluídos {dias_antes - len(matriz_perfis)} dias incompletos).")

datas_idx = pd.to_datetime(matriz_perfis.index)
meta_dias = pd.DataFrame({
    'Estacao': datas_idx.month.map(obter_estacao),
    'Tipo_Dia': np.where(datas_idx.dayofweek >= 5, 'Fim de Semana', 'Dia Útil')
}, index=matriz_perfis.index)

# 2. Padronização e PCA
X_scaled = StandardScaler().fit_transform(matriz_perfis)
pca = PCA(n_components=24, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Garantir orientação positiva de PC1 (PC1 alto = consumo alto)
if pca.components_[0].sum() < 0:
    pca.components_[0] *= -1
    X_pca[:, 0] *= -1

var_exp = pca.explained_variance_ratio_ * 100
var_cum = np.cumsum(var_exp)
print(f"Variância explicada -> PC1: {var_exp[0]:.2f}% | PC2: {var_exp[1]:.2f}% | PC3: {var_exp[2]:.2f}% (Acumulado: {var_cum[2]:.2f}%)")

# FIGURA 6: Scree Plot + Loadings
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
comps = np.arange(1, 25)
ax1.bar(comps, var_exp, color='#3498db', alpha=0.8, label='Variância Individual (%)')
ax1.plot(comps, var_cum, color='#e74c3c', marker='o', linewidth=2, label='Variância Acumulada (%)')
ax1.axhline(80, color='#2c3e50', linestyle='--', linewidth=1.2, label='Limiar 80%')
ax1.set_ylim(0, 105)
ax1.set_xticks(range(1, 25, 2))
ax1.set_title('Os primeiros 3 componentes concentram a estrutura dominante dos perfis diários', fontsize=11.5, fontweight='bold', loc='left')
ax1.set_xlabel('Componente Principal (PC1 – PC24)')
ax1.set_ylabel('Variância Explicada (%)')
ax1.legend(loc='center right')

horas = np.arange(24)
loadings = pca.components_[:3]
ax2.plot(horas, loadings[0], marker='o', linewidth=2.3, color='#1f618d', label=f'PC1 ({var_exp[0]:.1f}%) — Nível Global')
ax2.plot(horas, loadings[1], marker='s', linewidth=2.1, color='#d35400', label=f'PC2 ({var_exp[1]:.1f}%) — Dia vs. Noite')
ax2.plot(horas, loadings[2], marker='^', linewidth=1.9, color='#27ae60', label=f'PC3 ({var_exp[2]:.1f}%) — Manhã/Almoço')
ax2.axhline(0, color='black', linewidth=1, linestyle=':')
ax2.set_xticks(range(0, 24, 2))
ax2.set_title('Loadings: PC1 mede o volume total e PC2 opõe o período diurno ao noturno', fontsize=11.5, fontweight='bold', loc='left')
ax2.set_xlabel('Hora do Dia (0h – 23h)')
ax2.set_ylabel('Peso (Loading)')
ax2.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

# 3. t-SNE com seed fixa
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca', learning_rate='auto')
X_tsne = tsne.fit_transform(X_scaled)

# FIGURA 7: Comparação PCA vs t-SNE
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
paleta_estacoes = {'Inverno': '#2980b9', 'Primavera': '#27ae60', 'Verão': '#f39c12', 'Outono': '#8e44ad'}
paleta_dias = {'Dia Útil': '#34495e', 'Fim de Semana': '#e74c3c'}

for est in estacoes_ordem:
    mask = meta_dias['Estacao'] == est
    axes[0, 0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=est, color=paleta_estacoes[est], alpha=0.72, s=32)
    axes[1, 0].scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=est, color=paleta_estacoes[est], alpha=0.75, s=32)

for tipo in ['Dia Útil', 'Fim de Semana']:
    mask = meta_dias['Tipo_Dia'] == tipo
    axes[0, 1].scatter(X_pca[mask, 0], X_pca[mask, 1], label=tipo, color=paleta_dias[tipo], alpha=0.72, s=32)
    axes[1, 1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=tipo, color=paleta_dias[tipo], alpha=0.75, s=32)

axes[0, 0].set_title('PCA (PC1 × PC2) por Estação: Inverno e Verão separam-se ao longo de PC1', fontweight='bold', loc='left')
axes[0, 0].set_xlabel(f'PC1 ({var_exp[0]:.1f}% var.)')
axes[0, 0].set_ylabel(f'PC2 ({var_exp[1]:.1f}% var.)')
axes[0, 0].legend(title='Estação')

axes[0, 1].set_title('PCA (PC1 × PC2) por Tipo de Dia: Fins de semana deslocam-se para PC1/PC2 > 0', fontweight='bold', loc='left')
axes[0, 1].set_xlabel(f'PC1 ({var_exp[0]:.1f}% var.)')
axes[0, 1].set_ylabel(f'PC2 ({var_exp[1]:.1f}% var.)')
axes[0, 1].legend(title='Tipo de Dia')

axes[1, 0].set_title('t-SNE (seed=42) por Estação: Agrupa regimes locais mas perde distâncias globais', fontweight='bold', loc='left')
axes[1, 0].set_xlabel('Dimensão t-SNE 1 (sem unidade)')
axes[1, 0].set_ylabel('Dimensão t-SNE 2 (sem unidade)')
axes[1, 0].legend(title='Estação')

axes[1, 1].set_title('t-SNE (seed=42) por Tipo de Dia: Separa micro-clusters de rotinas semelhantes', fontweight='bold', loc='left')
axes[1, 1].set_xlabel('Dimensão t-SNE 1 (sem unidade)')
axes[1, 1].set_ylabel('Dimensão t-SNE 2 (sem unidade)')
axes[1, 1].legend(title='Tipo de Dia')

plt.tight_layout()
plt.show()

### Interpretação dos *Loadings* e Comparação PCA vs. t-SNE
- **O que é um `PC1` alto?** Como todos os 24 pesos são positivos, `PC1` mede a intensidade global de consumo diário (`PC1` alto = dia de forte consumo contínuo; `PC1` muito negativo = dias próximos da carga basal, como nas férias de agosto).
- **O que NÃO se pode ler no mapa t-SNE:** (1) Distâncias globais entre clusters afastados; (2) Tamanho ou dispersão visual dos grupos; (3) Significado físico ou unidades nos eixos.